### Imports 

In [15]:
import pandas as pd
import numpy as np
import joblib
import os
import seaborn as sns
import matplotlib.pyplot as plt
import itertools
from scipy.stats import t
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_selection import r_regression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score

In [4]:
import sys
print(sys.executable)

/home/user_stel/miniconda3/bin/python


### Loading the Data

In [16]:
path_to_dev = "../data/assignment1_dev_set.csv"
path_to_val = "../data/assignment1_val_set.csv"

# Function that retrieves data from a provided path
# and reads the data as a pandas dataframe
def load_data(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"The file at {path} was not found.")
    if 'dev' in path:
        print('Data for development:')
    elif 'val' in path:
        print('Data for evaluation:')
    else:
        raise ValueError("The path provided does not contain data for development nor for evaluation.")
    data = pd.read_csv(path)
    return data

dev_set_df=load_data(path_to_dev)
print(dev_set_df.head())
print(dev_set_df.info())
print(list(dev_set_df.columns))
val_set_df=load_data(path_to_val)
print(val_set_df.head())

Data for development:
   Unnamed: 0   Project ID Experiment type     Sex  Host age    BMI  \
0           0   PRJEB11419    Metagenomics    Male      53.0  19.01   
1           1  PRJNA388263    Metagenomics  Female      21.0  23.50   
2           2  PRJNA388263    Metagenomics    Male      52.0  25.80   
3           3   PRJEB11419    Metagenomics  Female      40.0  23.49   
4           4   PRJEB11419    Metagenomics  Female      30.0  22.60   

  Disease MESH ID  Acholeplasma axanthum  Acidaminococcus fermentans  \
0         D006262               0.000000                    0.000000   
1         D006262               0.001028                    0.000000   
2         D006262               0.001406                    0.000000   
3         D006262               0.000000                    0.008825   
4         D006262               0.002878                    0.037419   

   Acidaminococcus intestini  ...  Clostridium sphenoides  \
0                   0.000000  ...                0.005891

### Data Preprocessing

In [17]:
# Create pipeline for preprocessing 
def preprocess_data(df, output_path, columns_to_drop, scale=True):
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

    num_list = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_list = df.select_dtypes(exclude=[np.number]).columns.tolist()
    
    for column in num_list + cat_list:
        if column not in df.columns:
            raise ValueError(f"'{column}' could not be found in the dataframe provided.") 
    
    for col in cat_list:
        df[col] = LabelEncoder().fit_transform(df[col])

    if scale:
        num_pipeline=Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    else:
        num_pipeline=Pipeline([('imputer', SimpleImputer(strategy='mean'))])


    df[num_list] = num_pipeline.fit_transform(df[num_list])

    df.to_csv(output_path, index=False)

    print(f"The preprocessed data was saved to {output_path}.")
    return df

unnecessary_columns=['Unnamed: 0', 'Project ID', 'Experiment type', 'Disease MESH ID']
path_to_final_dev = "../data/development_final_data.csv"
path_to_final_val = "../data/evaluation_final_data.csv"

preprocess_data(dev_set_df, path_to_final_dev, unnecessary_columns, scale=True)
preprocess_data(val_set_df, path_to_final_val, unnecessary_columns, scale=True)

The preprocessed data was saved to ../data/development_final_data.csv.
The preprocessed data was saved to ../data/evaluation_final_data.csv.


,Sex,Host age,BMI,Acholeplasma axanthum,Acidaminococcus fermentans,Acidaminococcus intestini,Actinomyces lingnae,Akkermansia muciniphila,Alistipes finegoldii,Alistipes indistinctus,...,Clostridium sphenoides,Clostridium spiroforme,Clostridium stercorarium,Clostridium symbiosum,Clostridium thermosuccinogenes,Clostridium xylanolyticum,Eubacterium brachy,Eubacterium dolichum,Eubacterium sulci,Ruminococcus gnavus
0,1,0.855041,0.228555,-0.137066,-0.151394,1.507344,-0.123487,-0.340352,-0.235384,-0.150081,...,-0.379542,4.900793,-0.180919,-0.163163,0.634181,-0.161203,0.313723,-0.104544,-0.170046,0.635386
1,1,0.667761,0.603135,-0.137066,-0.151394,-0.389535,-0.123487,-0.340352,-0.320879,-0.237927,...,-0.379542,-0.429761,-0.180919,-0.488701,-0.233484,-0.236104,-0.465205,-0.104544,-0.434144,-0.199604
2,1,1.167176,-0.616134,-0.137066,-0.109337,-0.389535,-0.123487,-0.339793,-0.268389,-0.237927,...,-0.379542,-0.309204,-0.180919,-0.408756,-0.233484,-0.036834,0.311906,-0.051068,-0.065268,-0.199604
3,1,0.542907,0.180790,-0.137066,-0.151394,-0.336909,-0.123487,-0.336658,-0.307868,-0.237927,...,-0.322818,-0.262680,-0.180919,-0.472188,-0.181469,-0.236104,-0.465205,0.072187,0.436634,-0.178628
4,0,0.480480,-0.804680,-0.137066,-0.151394,-0.389535,-0.123487,-0.340352,-0.320879,-0.237927,...,-0.379542,-0.429761,-0.180919,-0.488701,-0.233484,-0.236104,-0.465205,-0.104544,-0.434144,-0.199604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,0,0.293199,-0.535687,-0.137066,-0.151394,0.419868,-0.123487,-0.340352,-0.320879,-0.237927,...,-0.213970,0.283027,-0.180919,-0.151299,-0.233484,-0.019846,-0.465205,-0.104544,-0.434144,-0.176644
207,1,0.480480,0.200902,-0.137066,-0.151394,-0.389535,-0.123487,-0.340352,-0.320879,-0.237927,...,-0.379542,-0.429761,-0.180919,-0.488701,-0.233484,-0.236104,-0.465205,-0.104544,-0.434144,-0.199604
208,1,0.355626,0.251181,-0.137066,-0.151394,-0.389535,-0.095980,-0.339189,-0.180760,-0.237927,...,-0.379542,-0.386618,-0.173805,-0.412483,-0.233484,-0.236104,-0.357440,-0.104544,-0.105303,-0.175400
209,1,-1.267473,0.437214,-0.137066,-0.151394,-0.370863,-0.123487,-0.284983,-0.320879,-0.237927,...,-0.292328,0.617565,-0.180919,-0.412534,-0.233484,0.118290,-0.465205,-0.104544,-0.434144,-0.167353


### Feature Selection: Supervised

In [4]:
dev_set_cleaned_df=preprocess_data(dev_set_df, path_to_final_dev, unnecessary_columns, scale=True)
val_set_cleaned_df=preprocess_data(val_set_df, path_to_final_val, unnecessary_columns, scale=True)

# Create a function that will select features based on the Pearson's Correlation Coefficient Method
def select_features(X, y, threshold):
    correlations=pd.Series(r_regression(X, y), index=X.columns)
    selected_features=correlations[correlations.abs() >= threshold].index
    reduced_df=X[selected_features]
    
    print(f"The selected features of {X.shape[1]} were: {len(selected_features)}")
    return reduced_df, correlations

X=dev_set_cleaned_df.drop(['BMI', 'Host age', 'Sex'], errors='ignore') # Bacteria Species -> features
y=dev_set_cleaned_df['BMI']

dev_set_selected_df, corr_scores=select_features(X, y, threshold=0.1)
print(dev_set_selected_df)


The preprocessed data was saved to ../data/development_final_data.csv.
The preprocessed data was saved to ../data/evaluation_final_data.csv.
The selected features of 137 were: 14
     Sex  Host age       BMI  Alistipes putredinis  Christensenella minuta  \
0      1  0.400741 -1.410654             -0.610922               -0.315060   
1      0 -1.635900 -0.342472             -0.426379                8.115256   
2      1  0.337096  0.204703             -0.564758                0.032101   
3      0 -0.426644 -0.344851             -0.341570               -0.275447   
4      0 -1.063095 -0.556584             -0.147897               -0.243995   
..   ...       ...       ...                   ...                     ...   
484    1  0.846257  0.007244              0.601130               -0.315060   
485    1  0.337096  0.204703             -0.508639               -0.283729   
486    1  1.991868  1.092079              0.185016                0.004534   
487    1 -0.235709  0.145227             

### Feature Selection: Unsupervised

In [ ]:
dev_set_cleaned_df=preprocess_data(dev_set_df, path_to_final_dev, unnecessary_columns, scale=False)

# Pearson's Correlation Coefficient for Bacteria species 
# Check for high correlation between bacteria species in order to exclude some of the species and reduce dimensionality 
bacteria_species=dev_set_cleaned_df.columns.drop(['BMI', 'Host age', 'Sex'])
correlation_matrix=dev_set_cleaned_df[bacteria_species].corr(method='pearson')

threshold = 0.5
high_corr_pairs = (
    correlation_matrix.abs()
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[high_corr_pairs > threshold]

print("Highly correlated pairs:")
print(high_corr_pairs)

# Species to remove (second species in each pair)
species_to_remove = set()

for pair in high_corr_pairs.index:
    species_to_remove.add(pair[1])

print("Species to remove:", species_to_remove)

# Drop highly correlated species
dev_set_selected_df = dev_set_cleaned_df.drop(columns=species_to_remove)
print(f"Reduced from {len(dev_set_cleaned_df.columns)} to {len(dev_set_selected_df.columns)} columns.")

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='Oranges', center=0)
plt.title('Correlation Heatmap of Bacterial Species')
plt.show()

### Principal Component Analysis

In [ ]:
pca = PCA(n_components=min(dev_set_cleaned_df.shape))
dev_set_cleaned_pca=pca.fit_transform(dev_set_cleaned_df)
explained_variances = pca.explained_variance_ratio_

# PCA explained variance ratio
print("Explained variance ratios:", explained_variances)

plt.figure(figsize=(10, 6))

# Cumulative explained variance plot
plt.plot(range(1, len(explained_variances) + 1), 
         explained_variances.cumsum(), 
         marker='o', linestyle='--')

plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Scree Plot: Explained Variance by PCA Components')
plt.grid(True)
plt.xticks(range(1, len(explained_variances) + 1))
plt.axhline(y=0.90, color='r', linestyle='-', label='90% variance explained')
plt.axhline(y=0.95, color='r', linestyle='-', label='95% variance explained')
plt.legend()
plt.tight_layout()
plt.show()


### Data Visualization

In [ ]:
# Example: BMI distribution plot
plt.figure(figsize=(8, 5))
sns.histplot(dev_set_cleaned_df['BMI'], kde=True)
plt.title("BMI Distribution")
plt.show()

# Correlation heatmap (clearly visualizes relationships)
plt.figure(figsize=(12, 10))
sns.heatmap(dev_set_cleaned_df.corr(), cmap='spring')
plt.title("Correlation Heatmap")
plt.show()

# Calculate mean abundance for each bacterial species across samples
bacteria_species_cleaned=dev_set_cleaned_df.columns.drop(['BMI', 'Host age', 'Sex'])
mean_abundance = dev_set_cleaned_df[bacteria_species_cleaned].mean().sort_values(ascending=False).head(15)

# Plotting the top 15 most abundant bacteria species
plt.figure(figsize=(12, 6))
sns.barplot(x=mean_abundance.index, y=mean_abundance.values, palette="viridis")

plt.title('Top 15 Most Abundant Bacterial Species')
plt.ylabel('Average Abundance (Z-score scaled)')
plt.xlabel('Bacterial Species')
plt.xticks(rotation=80, ha='right')
plt.tight_layout()
plt.show()

### Model Analysis 

In [8]:
# Generate baseline model with default parameters and no feature selection

df_features = dev_set_cleaned_df.drop(columns=['BMI', 'Host age', 'Sex'], errors='ignore')
df_target = dev_set_cleaned_df['BMI']

models={
    'enet':ElasticNet(),
    'svr':SVR(),
    'breg':BayesianRidge()
}

def train_model(models, df_features, df_target):
    results={}

    x_train, x_test, y_train, y_test = train_test_split(
    df_features, df_target, test_size=0.3, random_state=42)

    for model, model_instance in models.items():
        model_instance.fit(x_train, y_train)
        y_pred=model_instance.predict(x_test)
        rmse=root_mean_squared_error(y_test, y_pred)
        results[model]=rmse
        print(f"{model} RMSE: {rmse:.4f}")
    return results

print('Baseline Results: No feature selection, no model-tuning')
model_results = train_model(models, df_features, df_target)

Baseline Results: No feature selection, no model-tuning
enet RMSE: 3.8141
svr RMSE: 3.7522
breg RMSE: 3.8384


In [10]:
df_features = dev_set_selected_df.drop(columns=['BMI', 'Host age', 'Sex'], errors='ignore')
df_target = dev_set_selected_df['BMI']

print('Baseline Results: With selected features, no model-tuning')
model_results = train_model(models, df_features, df_target)

Baseline Results: With selected features, no model-tuning
enet RMSE: 3.8494
svr RMSE: 3.7490
breg RMSE: 3.8319


#### Model-tuning & Grid Search

In [11]:
# Model tuning using 
def cross_validate(models, df_features, df_target, cv=5):
    cv_results={}
    
    for model, model_instance in models.items():
        neg_mse_scores = cross_val_score(model_instance, X, y, scoring='neg_mean_squared_error', cv=cv)
        neg_mae_scores = cross_val_score(model_instance, X, y, scoring='neg_mean_absolute_error', cv=cv)

        rmse_scores = (-neg_mse_scores) ** 0.5
        mae_scores = -neg_mae_scores

        cv_results[model_instance] = {
            'Mean RMSE': rmse_scores.mean(),
            'Mean MAE': mae_scores.mean(),
            'Params': model_instance.get_params()
        }
        print(f"{model} - Mean RMSE: {rmse_scores.mean():.4f}, Mean MAE: {mae_scores.mean():.4f}, Params: {model_instance.get_params()}")

    return cv_results

selected_features = dev_set_selected_df.drop(columns=['BMI', 'Host age', 'Sex'], errors='ignore')
df_target = dev_set_selected_df['BMI']

model_cv_results=cross_validate(models, selected_features, df_target, cv=5)

enet - Mean RMSE: 0.6673, Mean MAE: 0.4398, Params: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'l1_ratio': 0.5, 'max_iter': 1000, 'positive': False, 'precompute': False, 'random_state': None, 'selection': 'cyclic', 'tol': 0.0001, 'warm_start': False}
svr - Mean RMSE: 0.5785, Mean MAE: 0.3005, Params: {'C': 1.0, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.1, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
breg - Mean RMSE: 0.0000, Mean MAE: 0.0000, Params: {'alpha_1': 1e-06, 'alpha_2': 1e-06, 'alpha_init': None, 'compute_score': False, 'copy_X': True, 'fit_intercept': True, 'lambda_1': 1e-06, 'lambda_2': 1e-06, 'lambda_init': None, 'max_iter': 300, 'tol': 0.001, 'verbose': False}


In [ ]:
# parameters 
param_grid = {
    'enet': {
        'alpha': [0.01, 0.1, 1.0],
        'l1_ratio': [0.2, 0.5, 0.8]
    },
    'svr': {
        'C': [0.1, 1, 10],
        'epsilon': [0.01, 0.1],
        'kernel': ['rbf', 'linear']
    },
    'breg': {
        'alpha_1': [1e-6, 1e-5],
        'lambda_1': [1e-6, 1e-5]
    }
}

# parameter combination for each model instance
enet_combinations = [
    dict(zip(param_grid['enet'].keys(), values))
    for values in itertools.product(*param_grid['enet'].values())
]

svr_combinations = [
    dict(zip(param_grid['svr'].keys(), values))
    for values in itertools.product(*param_grid['svr'].values())
]

breg_combinations = [
    dict(zip(param_grid['breg'].keys(), values))
    for values in itertools.product(*param_grid['breg'].values())
]

In [ ]:
model_combinations = {
    model_name: [
        dict(zip(params.keys(), values))
        for values in itertools.product(*params.values())
    ]
    for model_name, params in param_grid.items()
}

models={
    'enet':ElasticNet,
    'svr':SVR,
    'breg':BayesianRidge
}


def model_tuning(models, df_features, df_target, params_grids, cv):
    best_results={}

    for model, param_grid in param_grid.items():
        best_rmse=float('inf')
        best_model=None
        best_params=None 

        keys=params_grids.keys()
        values=param_grid.values()

        for combination in model_combinations[model]:
            model_instance=models[model](**combination)
            scores=cross_val_score(model_instance, df_features, df_target, scoring='neg_root_mean_squared_error', cv=cv)
            rmse=(-scores.mean())**0.5

            print(f"[{model}] Tested params: {combination}")
            print(f"[{model}] RMSE: {rmse:.4f}")

            if rmse < best_rmse:
                best_rmse = rmse
                best_model = model_instance
                best_params = combination
                print(f"[{model}] New best RMSE: {best_rmse:.4f}")
                print(f"[{model}] Best params so far: {best_params}")

        best_results[model] = {
            'Best RMSE': best_rmse,
            'Best Model': best_model,
            'Best Params': best_params
        }
    return best_results

model_analysis_final=model_tuning(models, selected_features, df_target, param_grid, cv=5)
print(model_analysis_final['enet'])


[enet] Tested params: {'alpha': 0.01, 'l1_ratio': 0.2}
[enet] RMSE: 2.1137
[enet] New best RMSE: 2.1137
[enet] Best params so far: {'alpha': 0.01, 'l1_ratio': 0.2}
[enet] Tested params: {'alpha': 0.01, 'l1_ratio': 0.5}
[enet] RMSE: 2.1244
[enet] Tested params: {'alpha': 0.01, 'l1_ratio': 0.8}
[enet] RMSE: 2.1379
[enet] Tested params: {'alpha': 0.1, 'l1_ratio': 0.2}
[enet] RMSE: 2.0607
[enet] New best RMSE: 2.0607
[enet] Best params so far: {'alpha': 0.1, 'l1_ratio': 0.2}
[enet] Tested params: {'alpha': 0.1, 'l1_ratio': 0.5}
[enet] RMSE: 2.0650
[enet] Tested params: {'alpha': 0.1, 'l1_ratio': 0.8}
[enet] RMSE: 2.0684
[enet] Tested params: {'alpha': 1.0, 'l1_ratio': 0.2}
[enet] RMSE: 2.0346
[enet] New best RMSE: 2.0346
[enet] Best params so far: {'alpha': 1.0, 'l1_ratio': 0.2}
[enet] Tested params: {'alpha': 1.0, 'l1_ratio': 0.5}
[enet] RMSE: 2.0366
[enet] Tested params: {'alpha': 1.0, 'l1_ratio': 0.8}
[enet] RMSE: 2.0374
[svr] Tested params: {'C': 0.1, 'epsilon': 0.01, 'kernel': 'rbf'}


In [46]:
print(model_analysis_final)

{'enet': {'Best RMSE': np.float64(4.206524075776147), 'Best Model': ElasticNet(l1_ratio=0.2), 'Best Params': {'alpha': 1.0, 'l1_ratio': 0.2}}, 'svr': {'Best RMSE': np.float64(4.1689797924256435), 'Best Model': SVR(C=1, epsilon=0.01), 'Best Params': {'C': 1, 'epsilon': 0.01, 'kernel': 'rbf'}}, 'breg': {'Best RMSE': np.float64(4.209226447895146), 'Best Model': BayesianRidge(), 'Best Params': {'alpha_1': 1e-06, 'lambda_1': 1e-06}}}


### Evaluation

In [52]:
def summarize(scores):
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    ci95 = t.interval(0.95, len(scores) - 1, loc=mean, scale=std / np.sqrt(len(scores)))
    return {
        'mean': mean,
        'median': np.median(scores),
        '95% CI': ci95
    }

In [ ]:
def evaluate_model(model, X, y, runs=30, test_size=0.2, save_path="../final_models/final_models.pkl"):
    metrics={
        'rmse':[],
        'mae':[],
        'r2':[]
    }

    for i in range(runs):
        X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=test_size)
        model.fit(X_train, y_train)
        y_pred=model.predict(X_test)

        metrics['rmse'].append(root_mean_squared_error(y_test, y_pred))
        metrics['mae'].append(mean_absolute_error(y_test, y_pred))
        metrics['r2'].append(r2_score(y_test, y_pred))

        results = {k: summarize(v) for k, v in metrics.items()}

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(model, f"{save_path}_{model}.pkl")
    print(f"Model saved to {save_path}")

    return results

In [63]:
# For the evaluation we are going to use a different dataset that was not previously used for the development of the model so 
# we have to make sure that the 
def align_evaluation_set(dev_df, val_df):
    dev_columns=dev_df.columns
    val_aligned=val_df.copy()
    val_aligned=val_aligned.reindex(columns=dev_columns, fill_value=0)
    print("Evaluation dataset was aligned to development feature set.")
    return val_aligned

In [67]:
val_final_df=align_evaluation_set(dev_set_selected_df, val_set_cleaned_df)
print(val_final_df)

Evaluation dataset was aligned to development feature set.
     Sex  Host age       BMI  Alistipes putredinis  Christensenella minuta  \
0      1  0.855041  0.228555             -0.310301               -0.430828   
1      1  0.667761  0.603135             -0.664596               -0.454995   
2      1  1.167176 -0.616134             -0.151691               -0.262109   
3      1  0.542907  0.180790             -0.639741               -0.454995   
4      0  0.480480 -0.804680             -0.664596               -0.454995   
..   ...       ...       ...                   ...                     ...   
206    0  0.293199 -0.535687             -0.388911               -0.454995   
207    1  0.480480  0.200902             -0.664596               -0.454995   
208    1  0.355626  0.251181             -0.590550               -0.073834   
209    1 -1.267473  0.437214              0.279305               -0.454995   
210    0 -1.704461 -0.487922              2.118868               -0.402503   

    

In [ ]:
X = val_final_df.drop(columns=['BMI', 'Host age', 'Sex'], errors='ignore')
y = val_final_df['BMI']

evaluate_model(model=ElasticNet(), X=X, y=y, runs=30, test_size=0.2, save_path="../final_models/final_models.pkl")
